Generates KDE plots of roughness values for damaged vs. undamaged coordinates, paneled per anvil.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Path to the folder containing one folder per island locality. Each locality folder contains one folder per anvil with an _output.txt file.
root_path = Path("FILEPATH")

random_seed = 4

# Load the data
records = []

for anvil_dir in sorted(root_path.glob("*/*")):
    if not anvil_dir.is_dir():
        continue

    anvil_id = anvil_dir.name

    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue

    output_file = output_files[0]
    print(f"Loading {anvil_id}...")

    with open(output_file, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break

    if "Roughness (10)" not in df_file.columns or "condition" not in df_file.columns:
        print(f"  Skipping {anvil_id} — missing expected columns")
        continue

    subset = df_file[["Roughness (10)", "Mean curvature (10)", "condition"]].dropna()
    subset = subset.rename(columns={"Roughness (10)": "roughness", "Mean curvature (10)": "curvature"})
    subset["anvil_id"] = anvil_id
    records.append(subset)
    print(f"  Loaded {len(subset):,} rows")

print("\nCombining all anvils...")
df = pd.concat(records, ignore_index=True)
if df.empty:
    raise RuntimeError("No roughness data loaded. Check paths, filenames, and column names.")

print(f"Total rows loaded: {len(df):,}")

print("\nAnvils and conditions found:")
for anvil in sorted(df["anvil_id"].unique()):
    conditions = df[df["anvil_id"] == anvil]["condition"].unique()
    print(f"  {anvil}: {', '.join(conditions)}")

# Balance sample sizes per anvil & condition for plotting
print("\nBalancing sample sizes...")
balanced_chunks = []

for anvil in df["anvil_id"].unique():
    sub_anvil = df[df["anvil_id"] == anvil]

    n_d = (sub_anvil["condition"] == "damaged").sum()
    n_u = (sub_anvil["condition"] == "undamaged").sum()

    if n_d == 0 or n_u == 0:
        balanced_chunks.append(sub_anvil)
        continue

    target_n = min(n_d, n_u)

    damaged_sub = sub_anvil[sub_anvil["condition"] == "damaged"]
    if len(damaged_sub) > target_n:
        damaged_sub = damaged_sub.sample(n=target_n, random_state=random_seed, replace=False)

    undamaged_sub = sub_anvil[sub_anvil["condition"] == "undamaged"]
    if len(undamaged_sub) > target_n:
        undamaged_sub = undamaged_sub.sample(n=target_n, random_state=random_seed, replace=False)

    balanced_chunks.append(pd.concat([damaged_sub, undamaged_sub], ignore_index=True))
    print(f"  {anvil}: n_damaged={n_d:,}, n_undamaged={n_u:,} → balanced to {target_n:,} each")

df_plot = pd.concat(balanced_chunks, ignore_index=True)
print(f"\nBalanced dataset: {len(df_plot):,} total rows")

count_table = (
    df.groupby(["anvil_id", "condition"])
      .size()
      .unstack(fill_value=0)
)

# Create panel plot
print("\nPlotting...")
sns.set_theme(style="whitegrid", context="talk")

anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]
anvils = [a for a in anvil_order if a in df_plot["anvil_id"].unique()]
n_anvils = len(anvils)

n_cols = 5
n_rows = 4

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(22, 12),
    sharex=False,
    sharey=False,
)
axes = axes.flatten()

condition_colors = {
    "undamaged": "#1D9E75",
    "damaged":   "#D85A30",
}

for i, anvil in enumerate(anvils):
    ax = axes[i]
    grp = df_plot[df_plot["anvil_id"] == anvil]

    for condition in ["undamaged", "damaged"]:
        sub = grp[grp["condition"] == condition]
        if sub.empty:
            continue
        sns.kdeplot(
            data=sub, x="roughness", ax=ax,
            label=condition, fill=True, alpha=0.3,
            linewidth=1.5, color=condition_colors.get(condition),
        )

    ax.set_title(anvil)
    ax.set_xlim(-0.1, 2.5)
    ax.set_ylabel("")
    ax.set_xlabel("")

    damaged_n   = len(grp[grp["condition"] == "damaged"])
    undamaged_n = len(grp[grp["condition"] == "undamaged"])
    info_text   = f"n damaged = {damaged_n:,}   |   n undamaged = {undamaged_n:,}"
    ax.text(0.5, -0.12, info_text, transform=ax.transAxes,
            ha="center", va="top", fontsize=7)

for ax in axes[:len(anvils)]:
    ax.tick_params(axis='x', pad=20)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.text(0.5, 0.04, 'Roughness [mm]', ha='center', fontsize=16)
fig.text(0.04, 0.5, 'Density', va='center', rotation='vertical', fontsize=16)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="center left", bbox_to_anchor=(0.92, 0.5),
    frameon=True, fancybox=True, shadow=True, fontsize=12
)

plt.tight_layout(rect=[0.06, 0.06, 0.90, 0.98])
print("Done.")
plt.show()

Generates KDE plots of curvature values for damaged vs. undamaged coordinates, paneled per anvil.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Path to the folder containing one folder per island locality. Each locality folder contains one folder per anvil with an _output.txt file.
root_path = Path("PATH")

random_seed = 4

# Load the data
records = []

for anvil_dir in sorted(root_path.glob("*/*")):
    if not anvil_dir.is_dir():
        continue

    anvil_id = anvil_dir.name

    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue

    output_file = output_files[0]
    print(f"Loading {anvil_id}...")

    with open(output_file, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break

    if "Mean curvature (10)" not in df_file.columns or "condition" not in df_file.columns:
        print(f"  Skipping {anvil_id} — missing expected columns")
        continue

print("\nCombining all anvils...")
records = []

for anvil_dir in sorted(root_path.iterdir()):
    if not anvil_dir.is_dir():
        continue

    anvil_id = anvil_dir.name
    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        continue

    output_file = output_files[0]
    with open(output_file, "r") as f:
        lines = f.readlines()

    df_file = None
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break

    if df_file is None:
        continue

    if "Mean curvature (10)" not in df_file.columns or "condition" not in df_file.columns:
        continue

    df_file = df_file.rename(columns={"Mean curvature (10)": "curvature", "Roughness (10)": "roughness"})
    if "roughness" not in df_file.columns:
        continue

    df_file["anvil_id"] = anvil_id
    df_file = df_file[["roughness", "curvature", "condition", "anvil_id"]]
    records.append(df_file)

df = pd.concat(records, ignore_index=True) if records else pd.DataFrame(
    columns=["roughness", "curvature", "condition", "anvil_id"]
)
if df.empty:
    raise RuntimeError("No curvature data loaded. Check paths, filenames, and column names.")

print(f"Total rows loaded: {len(df):,}")

print("\nAnvils and conditions found:")
for anvil in sorted(df["anvil_id"].unique()):
    conditions = df[df["anvil_id"] == anvil]["condition"].unique()
    print(f"  {anvil}: {', '.join(conditions)}")

# Balance sample sizes per anvil & condition for plotting
print("\nBalancing sample sizes...")
balanced_chunks = []

for anvil in df["anvil_id"].unique():
    sub_anvil = df[df["anvil_id"] == anvil]

    n_d = (sub_anvil["condition"] == "damaged").sum()
    n_u = (sub_anvil["condition"] == "undamaged").sum()

    if n_d == 0 or n_u == 0:
        balanced_chunks.append(sub_anvil)
        continue

    target_n = min(n_d, n_u)

    damaged_sub = sub_anvil[sub_anvil["condition"] == "damaged"]
    if len(damaged_sub) > target_n:
        damaged_sub = damaged_sub.sample(n=target_n, random_state=random_seed, replace=False)

    undamaged_sub = sub_anvil[sub_anvil["condition"] == "undamaged"]
    if len(undamaged_sub) > target_n:
        undamaged_sub = undamaged_sub.sample(n=target_n, random_state=random_seed, replace=False)

    balanced_chunks.append(pd.concat([damaged_sub, undamaged_sub], ignore_index=True))
    print(f"  {anvil}: n_damaged={n_d:,}, n_undamaged={n_u:,} → balanced to {target_n:,} each")

df_plot = pd.concat(balanced_chunks, ignore_index=True)
print(f"\nBalanced dataset: {len(df_plot):,} total rows")

count_table = (
    df.groupby(["anvil_id", "condition"])
      .size()
      .unstack(fill_value=0)
)

# Create panel plot
print("\nPlotting...")
sns.set_theme(style="whitegrid", context="talk")

anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]
anvils = [a for a in anvil_order if a in df_plot["anvil_id"].unique()]
n_anvils = len(anvils)

n_cols = 5
n_rows = 4

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(22, 12),
    sharex=False,
    sharey=False,
)
axes = axes.flatten()

condition_colors = {
    "undamaged": "#1D9E75",
    "damaged":   "#D85A30",
}

for i, anvil in enumerate(anvils):
    ax = axes[i]
    grp = df_plot[df_plot["anvil_id"] == anvil]

    for condition in ["undamaged", "damaged"]:
        sub = grp[grp["condition"] == condition]
        if sub.empty:
            continue
        sns.kdeplot(
            data=sub, x="curvature", ax=ax,
            label=condition, fill=True, alpha=0.3,
            linewidth=1.5, color=condition_colors.get(condition),
        )

    ax.set_title(anvil)
    ax.set_xlim(-0.01, 0.08)
    ax.set_ylabel("")
    ax.set_xlabel("")

    damaged_n   = len(grp[grp["condition"] == "damaged"])
    undamaged_n = len(grp[grp["condition"] == "undamaged"])
    info_text   = f"n damaged = {damaged_n:,}   |   n undamaged = {undamaged_n:,}"
    ax.text(0.5, -0.12, info_text, transform=ax.transAxes,
            ha="center", va="top", fontsize=7)

for ax in axes[:len(anvils)]:
    ax.tick_params(axis='x', pad=20)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.text(0.5, 0.04, 'Mean curvature [mm^-1]', ha='center', fontsize=16)
fig.text(0.04, 0.5, 'Density', va='center', rotation='vertical', fontsize=16)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="center left", bbox_to_anchor=(0.92, 0.5),
    frameon=True, fancybox=True, shadow=True, fontsize=12
)

plt.tight_layout(rect=[0.06, 0.06, 0.90, 0.98])
print("Done.")
plt.show()

Builds a summary table of the median and maximum roughness and curvature values per anvil and condition.

In [ ]:
summary_records = []

for anvil in sorted(df_plot["anvil_id"].unique()):
    sub = df_plot[df_plot["anvil_id"] == anvil]

    for condition in ["damaged", "undamaged"]:
        cond_sub = sub[sub["condition"] == condition]
        if cond_sub.empty:
            continue

        rough = cond_sub["roughness"]
        curv  = cond_sub["curvature"]

        summary_records.append({
            "anvil_id":         anvil,
            "condition":        condition,
            "median_roughness": rough.median(),
            "max_roughness":    rough.max(),
            "median_curvature": curv.median(),
            "max_curvature":    curv.max(),
        })

summary_df = pd.DataFrame.from_records(summary_records)

Creates a dot plot showing median roughness per anvil, split by condition.

In [ ]:
# Define custom order to group by location and raw material
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]

anvils_present = [a for a in anvil_order if a in summary_df["anvil_id"].unique()]

sns.set_theme(style="white", context="talk")

# Pivot the table
median_table = summary_df.pivot(
    index="anvil_id",
    columns="condition",
    values="median_roughness"
)

# Reindex to apply custom order.
median_table = median_table.reindex(anvils_present)

fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(
    median_table.index,
    median_table["undamaged"],
    color="#1D9E75",
    s=80,
    label="undamaged",
    zorder=2,
    alpha=0.8
)

ax.scatter(
    median_table.index,
    median_table["damaged"],
    color="#D85A30",
    s=80,
    label="damaged",
    zorder=2,
    alpha=0.8
)

ax.set_ylabel("Roughness [mm]", fontsize=12)
ax.set_xlabel("Anvil", fontsize=12)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45, ha='right')
ax.set_title("Median roughness by anvil", fontsize=14, weight='normal')

ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

ax.legend(fontsize=11, frameon=False, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

Creates a dot plot showing median curvature per anvil, split by condition.

In [ ]:
# Define custom order to group by location and raw material
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]

anvils_present = [a for a in anvil_order if a in summary_df["anvil_id"].unique()]

sns.set_theme(style="white", context="talk")

# Pivot the table
median_table = summary_df.pivot(
    index="anvil_id",
    columns="condition",
    values="median_curvature"
)

# Reindex to apply custom order
median_table = median_table.reindex(anvils_present)

fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(
    median_table.index,
    median_table["undamaged"],
    color="#1D9E75",
    s=80,
    label="undamaged",
    zorder=2,
    alpha=0.8
)

ax.scatter(
    median_table.index,
    median_table["damaged"],
    color="#D85A30",
    s=80,
    label="damaged",
    zorder=2,
    alpha=0.8
)

ax.set_ylabel("Median curvature [mm^-1]", fontsize=12)
ax.set_xlabel("Anvil", fontsize=12)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45, ha='right')
ax.set_title("Median curvature by anvil", fontsize=14, weight='normal')

ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.3f}'))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

ax.legend(fontsize=11, frameon=False, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

Creates a dot plot showing maximum roughness per anvil, split by condition.

In [ ]:
# Define custom order to group by location and raw material
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]

anvils_present = [a for a in anvil_order if a in summary_df["anvil_id"].unique()]

sns.set_theme(style="white", context="talk")

max_table = summary_df.pivot(
    index="anvil_id",
    columns="condition",
    values="max_roughness"
)

# Reindex to apply custom order
max_table = max_table.reindex(anvils_present)

fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(
    max_table.index,
    max_table["undamaged"],
    color="#1D9E75",
    s=80,
    label="undamaged",
    zorder=2,
    alpha=0.8
)

ax.scatter(
    max_table.index,
    max_table["damaged"],
    color="#D85A30",
    s=80,
    label="damaged",
    zorder=2,
    alpha=0.8
)

ax.set_ylabel("Roughness [mm]", fontsize=12)
ax.set_xlabel("Anvil", fontsize=12)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45, ha='right')
ax.set_title("Max roughness by anvil", fontsize=14, weight='normal')
ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

ax.legend(fontsize=11, frameon=False, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

Creates a dot plot showing maximum curvature per anvil, split by condition.

In [ ]:
# Define custom order to group by location and raw material
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]

anvils_present = [a for a in anvil_order if a in summary_df["anvil_id"].unique()]

sns.set_theme(style="white", context="talk")

max_table = summary_df.pivot(
    index="anvil_id",
    columns="condition",
    values="max_curvature"
)

# Reindex to apply custom order
max_table = max_table.reindex(anvils_present)

fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(
    max_table.index,
    max_table["undamaged"],
    color="#1D9E75",
    s=80,
    label="undamaged",
    zorder=2,
    alpha=0.8
)

ax.scatter(
    max_table.index,
    max_table["damaged"],
    color="#D85A30",
    s=80,
    label="damaged",
    zorder=2,
    alpha=0.8
)

ax.set_ylabel("Max curvature [mm^-1]", fontsize=12)
ax.set_xlabel("Anvil", fontsize=12)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45, ha='right')
ax.set_title("Max curvature by anvil", fontsize=14, weight='normal')
ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.3f}'))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

ax.legend(fontsize=11, frameon=False, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

Creates a boxplot of roughness grouped by locality and raw material.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Path to the folder containing one folder per island locality. Each locality folder contains one folder per anvil with an _output.txt file.
root_path = Path("FILEPATH")

random_seed = 4

# Load data
records = []
for anvil_dir in sorted(root_path.glob("*/*")):
    if not anvil_dir.is_dir():
        continue

    anvil_id = anvil_dir.name
    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue
    output_file = output_files[0]
    with open(output_file, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break
    if "Roughness (10)" not in df_file.columns or "condition" not in df_file.columns:
        print(f"  Skipping {anvil_id} — missing expected columns")
        continue
    subset = df_file[["Roughness (10)", "Mean curvature (10)", "condition"]].dropna()
    subset = subset.rename(columns={"Roughness (10)": "roughness", "Mean curvature (10)": "curvature"})
    subset["anvil_id"] = anvil_id
    records.append(subset)

df = pd.concat(records, ignore_index=True)

# Balance sample sizes per anvil & condition for plotting
balanced_chunks = []
for anvil in df["anvil_id"].unique():
    sub_anvil = df[df["anvil_id"] == anvil]
    n_d = (sub_anvil["condition"] == "damaged").sum()
    n_u = (sub_anvil["condition"] == "undamaged").sum()
    if n_d == 0 or n_u == 0:
        balanced_chunks.append(sub_anvil)
        continue
    target_n = min(n_d, n_u)
    damaged_sub = sub_anvil[sub_anvil["condition"] == "damaged"]
    if len(damaged_sub) > target_n:
        damaged_sub = damaged_sub.sample(n=target_n, random_state=random_seed, replace=False)
    undamaged_sub = sub_anvil[sub_anvil["condition"] == "undamaged"]
    if len(undamaged_sub) > target_n:
        undamaged_sub = undamaged_sub.sample(n=target_n, random_state=random_seed, replace=False)
    balanced_chunks.append(pd.concat([damaged_sub, undamaged_sub], ignore_index=True))

df_plot = pd.concat(balanced_chunks, ignore_index=True)

# Anvil order and grouping for plotting
group_map = {
    "KBN-AV2":      "KBN\nsandstone",
    "KBN-AV3":      "KBN\nsandstone",
    "KBN-AV4":      "KBN\nsandstone",
    "KBN-AV5":      "KBN\nsandstone",
    "KBN-AV6":      "KBN\nsandstone",
    "KBY-AV7":      "KBY\nsandstone",
    "KBY-AV8":      "KBY\nsandstone",
    "KBY-AV9":      "KBY\nsandstone",
    "KBY-AV11":     "KBY\nsandstone",
    "KBY-AV12":     "KBY\nsandstone",
    "AL-AV2":       "AL limestone\ntype I",
    "AL-AV3":       "AL limestone\ntype I",
    "AL-AV5_east":  "AL limestone\ntype I",
    "AL-AV5_west":  "AL limestone\ntype I",
    "AL-AV8":       "AL limestone\ntype I",
    "AL-AV7":       "AL limestone\ntype II",
    "AL-AV9":       "AL limestone\ntype II",
    "AL-AV12":      "AL limestone\ntype II",
    "AL-AV13":      "AL limestone\ntype II",
}

group_order = [
    "KBN\nsandstone",
    "KBY\nsandstone",
    "AL limestone\ntype I",
    "AL limestone\ntype II",
]

df_plot["group"] = df_plot["anvil_id"].map(group_map)
df_plot = df_plot[df_plot["group"].notna()]

# Print sample sizes per group and condition
print("\nSample sizes per group and condition:")
for group in group_order:
    grp = df_plot[df_plot["group"] == group]
    for condition in ["undamaged", "damaged"]:
        n = (grp["condition"] == condition).sum()
        print(f"  {group.replace(chr(10), ' '):<25} {condition}: n = {n}")

# Plot
sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(1, 4, figsize=(4 * 4, 5), sharey=True)

upper_limit = df_plot["roughness"].quantile(0.997)

for i, (group, ax) in enumerate(zip(group_order, axes)):
    grp = df_plot[df_plot["group"] == group]

    sns.boxplot(
        data=grp,
        x="condition",
        y="roughness",
        ax=ax,
        palette={"undamaged": "#1D9E75", "damaged": "#D85A30"},
        linewidth=1.5,
        flierprops={'marker': '_', 'markeredgewidth': 0.02, 'markersize': 4, 'alpha': 0.5},
        order=["undamaged", "damaged"],
    )

    ax.set_title(group, fontsize=12, weight='normal')
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([])
    ax.set_ylim(-0.1, upper_limit)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(i == 0)
    ax.tick_params(axis='y', left=(i == 0), length=0)

    ax.grid(axis='y', alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='both', labelsize=10)

fig.text(0.04, 0.5, 'Roughness [mm]', va='center', rotation='vertical', fontsize=16)

legend_elements = [
    mpatches.Patch(facecolor='#D85A30', label='damaged'),
    mpatches.Patch(facecolor='#1D9E75', label='undamaged'),
]
fig.legend(
    handles=legend_elements,
    loc="center left",
    bbox_to_anchor=(0.92, 0.5),
    frameon=True,
    fancybox=True,
    shadow=True,
    fontsize=12,
)

plt.tight_layout(rect=[0.06, 0.02, 0.90, 0.98])
plt.show()

Creates a boxplot of curvature grouped by locality and raw material.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Path to the folder containing one folder per island locality. Each locality folder contains one folder per anvil with an _output.txt file.
root_path = Path("FILEPATH")

random_seed = 4

# Load data
records = []
for anvil_dir in sorted(root_path.glob("*/*")):
    if not anvil_dir.is_dir():
        continue

    anvil_id = anvil_dir.name
    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue
    output_file = output_files[0]
    with open(output_file, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break
    if "Mean curvature (10)" not in df_file.columns or "condition" not in df_file.columns:
        print(f"  Skipping {anvil_id} — missing expected columns")
        continue
    subset = df_file[["Roughness (10)", "Mean curvature (10)", "condition"]].dropna()
    subset = subset.rename(columns={"Roughness (10)": "roughness", "Mean curvature (10)": "curvature"})
    subset["anvil_id"] = anvil_id
    records.append(subset)

df = pd.concat(records, ignore_index=True)

# Balance sample sizes per anvil & condition
balanced_chunks = []
for anvil in df["anvil_id"].unique():
    sub_anvil = df[df["anvil_id"] == anvil]
    n_d = (sub_anvil["condition"] == "damaged").sum()
    n_u = (sub_anvil["condition"] == "undamaged").sum()
    if n_d == 0 or n_u == 0:
        balanced_chunks.append(sub_anvil)
        continue
    target_n = min(n_d, n_u)
    damaged_sub = sub_anvil[sub_anvil["condition"] == "damaged"]
    if len(damaged_sub) > target_n:
        damaged_sub = damaged_sub.sample(n=target_n, random_state=random_seed, replace=False)
    undamaged_sub = sub_anvil[sub_anvil["condition"] == "undamaged"]
    if len(undamaged_sub) > target_n:
        undamaged_sub = undamaged_sub.sample(n=target_n, random_state=random_seed, replace=False)
    balanced_chunks.append(pd.concat([damaged_sub, undamaged_sub], ignore_index=True))

df_plot = pd.concat(balanced_chunks, ignore_index=True)

# Anvil order and grouping for plotting
group_map = {
    "KBN-AV2":      "KBN\nsandstone",
    "KBN-AV3":      "KBN\nsandstone",
    "KBN-AV4":      "KBN\nsandstone",
    "KBN-AV5":      "KBN\nsandstone",
    "KBN-AV6":      "KBN\nsandstone",
    "KBY-AV7":      "KBY\nsandstone",
    "KBY-AV8":      "KBY\nsandstone",
    "KBY-AV9":      "KBY\nsandstone",
    "KBY-AV11":     "KBY\nsandstone",
    "KBY-AV12":     "KBY\nsandstone",
    "AL-AV2":       "AL limestone\ntype I",
    "AL-AV3":       "AL limestone\ntype I",
    "AL-AV5_east":  "AL limestone\ntype I",
    "AL-AV5_west":  "AL limestone\ntype I",
    "AL-AV8":       "AL limestone\ntype I",
    "AL-AV7":       "AL limestone\ntype II",
    "AL-AV9":       "AL limestone\ntype II",
    "AL-AV12":      "AL limestone\ntype II",
    "AL-AV13":      "AL limestone\ntype II",
}

group_order = [
    "KBN\nsandstone",
    "KBY\nsandstone",
    "AL limestone\ntype I",
    "AL limestone\ntype II",
]

df_plot["group"] = df_plot["anvil_id"].map(group_map)
df_plot = df_plot[df_plot["group"].notna()]

# Print sample sizes per group and condition
print("\nSample sizes per group and condition:")
for group in group_order:
    grp = df_plot[df_plot["group"] == group]
    for condition in ["undamaged", "damaged"]:
        n = (grp["condition"] == condition).sum()
        print(f"  {group.replace(chr(10), ' '):<25} {condition}: n = {n}")

# Plot
sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(1, 4, figsize=(4 * 4, 5), sharey=True)

upper_limit = df_plot["curvature"].quantile(0.997)

for i, (group, ax) in enumerate(zip(group_order, axes)):
    grp = df_plot[df_plot["group"] == group]

    sns.boxplot(
        data=grp,
        x="condition",
        y="curvature",
        ax=ax,
        palette={"undamaged": "#1D9E75", "damaged": "#D85A30"},
        linewidth=1.5,
        flierprops={'marker': '_', 'markeredgewidth': 0.02, 'markersize': 4, 'alpha': 0.5},
        order=["undamaged", "damaged"],
    )

    ax.set_title(group, fontsize=12, weight='normal')
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([])
    ax.set_ylim(-0.005, upper_limit)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if i > 0:
        ax.spines['left'].set_visible(False)
        ax.tick_params(axis='y', left=False)

    ax.grid(axis='y', alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='both', labelsize=10)

fig.text(0.04, 0.5, 'Mean Curvature [mm⁻¹]', va='center', rotation='vertical', fontsize=16)

legend_elements = [
    mpatches.Patch(facecolor='#D85A30', label='damaged'),
    mpatches.Patch(facecolor='#1D9E75', label='undamaged'),
]
fig.legend(
    handles=legend_elements,
    loc="center left",
    bbox_to_anchor=(0.92, 0.5),
    frameon=True,
    fancybox=True,
    shadow=True,
    fontsize=12,
)

plt.tight_layout(rect=[0.06, 0.02, 0.90, 0.98])
plt.show()

Computes summary statistics and runs non-parametric tests (Mann-Whitney, Kruskal-Wallis) on roughness and curvature data across anvils and conditions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Configuration. The root path should point to the directory containing subdirectories for each anvil, each with an _spatial_join.txt file containing the scalar field (roughness, curvature)data for damaged and undamaged coordinates. 
root_path   = Path("FILEPATH")

random_seed = 4

group_map = {
    "KBN-AV2":      "KBN_sandstone",
    "KBN-AV3":      "KBN_sandstone",
    "KBN-AV4":      "KBN_sandstone",
    "KBN-AV5":      "KBN_sandstone",
    "KBN-AV6":      "KBN_sandstone",
    "KBY-AV7":      "KBY_sandstone",
    "KBY-AV8":      "KBY_sandstone",
    "KBY-AV9":      "KBY_sandstone",
    "KBY-AV11":     "KBY_sandstone",
    "KBY-AV12":     "KBY_sandstone",
    "AL-AV2":       "AL_limestone_I",
    "AL-AV3":       "AL_limestone_I",
    "AL-AV5_east":  "AL_limestone_I",
    "AL-AV5_west":  "AL_limestone_I",
    "AL-AV8":       "AL_limestone_I",
    "AL-AV7":       "AL_limestone_II",
    "AL-AV9":       "AL_limestone_II",
    "AL-AV12":      "AL_limestone_II",
    "AL-AV13":      "AL_limestone_II",
}

anvil_order = list(group_map.keys())

# Scalar fields to analyse
scalar_fields = {
    "roughness": "Roughness (10)",
    "curvature": "Mean curvature (10)",
}

# Data loading
records = []
for anvil_dir in sorted(root_path.glob("*/*")):
    if not anvil_dir.is_dir():
        continue
    anvil_id = anvil_dir.name

    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no _output.txt found")
        continue

    output_file = output_files[0]

    with open(output_file, "r") as f: # Assumes the first line with column names starts with "//", corresponding to CC format.
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break

     # Check for required columns
    missing = [col for col in list(scalar_fields.values()) + ["condition"] if col not in df_file.columns]
    if missing:
        print(f"Skipping {anvil_id} — missing columns: {missing}")
        continue

    subset = df_file[list(scalar_fields.values()) + ["condition"]].dropna()
    subset["anvil_id"] = anvil_id 
    records.append(subset)

df = pd.concat(records, ignore_index=True)
if df.empty:
    raise RuntimeError("No data loaded. Check paths and filenames.")

df["RM/locality"] = df["anvil_id"].map(group_map)
anvils = [a for a in anvil_order if a in df["anvil_id"].unique()]

# Balancing loop: downsample damaged n to match undamaged n per anvil
balanced_chunks = []

for anvil in anvils:
    sub       = df[df["anvil_id"] == anvil] 
    damaged   = sub[sub["condition"] == "damaged"]
    undamaged = sub[sub["condition"] == "undamaged"]

    if damaged.empty or undamaged.empty:
        print(f"Skipping {anvil} in balancing: missing one condition.")
        continue

    n_balanced    = min(len(damaged), len(undamaged))
    damaged_bal   = damaged.sample(n=n_balanced, random_state=random_seed)
    undamaged_bal = undamaged.sample(n=n_balanced, random_state=random_seed)

    balanced_chunks.append(damaged_bal)
    balanced_chunks.append(undamaged_bal)

    print(f"  {anvil}: n_damaged={len(damaged):,}, n_undamaged={len(undamaged):,} → balanced to {n_balanced:,} each")

# Combine all balanced chunks into a single DataFrame for analysis
df_balanced = pd.concat(balanced_chunks, ignore_index=True) 
print(f"\nBalanced dataset: {len(df_balanced):,} total rows.")

# Helper functions
def mann_whitney_cles(damaged, undamaged): # Returns U statistic, p-value, CLES, and group sizes for two independent samples using Mann-Whitney U test from scipy.
    u_stat, p_value = mannwhitneyu(damaged, undamaged, alternative="two-sided")
    n1, n2 = len(damaged), len(undamaged)
    cles = u_stat / (n1 * n2)
    return u_stat, p_value, cles, n1, n2

groups = ["KBN_sandstone", "KBY_sandstone", "AL_limestone_I", "AL_limestone_II"]

def between_group_analysis(df_input, value_col, label):
    group_data = {g: df_input[df_input["RM/locality"] == g][value_col].dropna().values
                  for g in groups}

    group_list = list(group_data.keys())

    pairs = [(group_list[i], group_list[j])
             for i in range(len(group_list)) for j in range(i + 1, len(group_list))]

    pair_rows = []

    for g1, g2 in pairs:
        u, p, cles, n1, n2 = mann_whitney_cles(group_data[g1], group_data[g2])
        pair_rows.append({
            "comparison":      label,
            "RM/locality_1":   g1,
            "RM/locality_2":   g2,
            "n_RM/locality_1": n1,
            "n_RM/locality_2": n2,
            "U":               u,
            "CLES":            cles,
        })
    pairwise_df = pd.DataFrame(pair_rows) 

    return pairwise_df

def run_analysis(df, field_col, anvils, group_map):
    """Run full analysis pipeline for one scalar field. Returns (summary_df, within_df, pairwise_df)."""

    summary_records = []

    for anvil in anvils:
        sub = df[df["anvil_id"] == anvil]
        for condition in ["damaged", "undamaged"]:
            cond_sub = sub[sub["condition"] == condition]
            if cond_sub.empty:
                continue
            values = cond_sub[field_col]
            summary_records.append({
                "anvil_id":    anvil,
                "RM/locality": group_map.get(anvil, ""),
                "condition":   condition,
                "n":           len(values),
                "mean":        values.mean(),
                "median":      values.median(),
                "std":         values.std(),
                "min":         values.min(),
                "max":         values.max(),
            })
    summary_df = pd.DataFrame(summary_records)

    # Within-anvil Mann-Whitney
    within_records = []

    for anvil in anvils:
        sub       = df[df["anvil_id"] == anvil]
        damaged   = sub[sub["condition"] == "damaged"][field_col]
        undamaged = sub[sub["condition"] == "undamaged"][field_col]
        if damaged.empty or undamaged.empty:
            print(f"  Skipping {anvil}: missing one condition.")
            continue
        u, p, cles, n1, n2 = mann_whitney_cles(damaged, undamaged)
        within_records.append({
            "anvil_id":    anvil,
            "RM/locality": group_map.get(anvil, ""),
            "n_damaged":   n1,
            "n_undamaged": n2,
            "U":           u,
            "CLES":        cles,
        })
    within_df = pd.DataFrame(within_records)

    # Between-group pairwise (damaged and undamaged)
    df_damaged   = df[df["condition"] == "damaged"]
    df_undamaged = df[df["condition"] == "undamaged"]

    pw_damaged   = between_group_analysis(df_damaged,   field_col, "damaged")
    pw_undamaged = between_group_analysis(df_undamaged, field_col, "undamaged")

    pairwise_df = pd.concat([pw_damaged, pw_undamaged], ignore_index=True)
    return summary_df, within_df, pairwise_df

# Excel style helpers
header_fill = PatternFill("solid", start_color="D6DCE4", end_color="D6DCE4")
sig_fill    = PatternFill("solid", start_color="E2EFDA", end_color="E2EFDA")
header_font = Font(name="Arial", bold=True, color="000000", size=10)
body_font   = Font(name="Arial", size=10)
center      = Alignment(horizontal="center", vertical="center")
thin_border = Border(
    bottom=Side(style="thin", color="BFBFBF"),
    right=Side(style="thin",  color="BFBFBF"),
)

def style_header_row(ws, row, n_cols):
    for col in range(1, n_cols + 1):
        cell = ws.cell(row=row, column=col)
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center

def style_body_row(ws, row, n_cols):
    for col in range(1, n_cols + 1):
        cell = ws.cell(row=row, column=col)
        cell.font      = body_font
        cell.alignment = center
        cell.border    = thin_border

def write_df_to_sheet(ws, df_in):
    header_row = ws.max_row
    col_names  = list(df_in.columns)
    ws.append(col_names)
    style_header_row(ws, header_row, len(col_names))

    for _, row_data in df_in.iterrows():
        ws.append(list(row_data))
        data_row = ws.max_row
        style_body_row(ws, data_row, len(col_names))

    for col_idx, col_name in enumerate(col_names, start=1):
        max_len = max(len(str(col_name)), df_in[col_name].astype(str).str.len().max())
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 4, 40)

    ws.append([])

# Run analysis and export
wb          = Workbook()
first_sheet = True

for field_name, field_col in scalar_fields.items():
    print(f"\nRunning analysis for: {field_name}")
    summary_df, within_df, pairwise_df = run_analysis(df_balanced, field_col, anvils, group_map)

    if first_sheet:
        ws = wb.active
        first_sheet = False
    else:
        ws = wb.create_sheet()

    ws.title = f"Summary - {field_name}"
    write_df_to_sheet(ws, summary_df)

    ws = wb.create_sheet(f"Within-Anvil MW - {field_name}")
    write_df_to_sheet(ws, within_df)

    ws = wb.create_sheet(f"Between-Group Pairwise - {field_name}")
    write_df_to_sheet(ws, pairwise_df)

output_path = root_path / "statistical_analysis.xlsx"
df_balanced.to_csv(root_path / "df_balanced.csv", index=False)
wb.save(output_path)
print(f"\nDone. Results saved to:\n  {output_path}")

Plots within-anvil effect sizes (CLES) for roughness and curvature as a dumbbell plot.

In [ ]:
# Path to "statistical_analysis.xlsx" file containing the within-anvil CLES data for roughness and curvature.
file_path = "FILEPATH"

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

# Load CLES data
within_df_roughness = pd.read_excel(file_path, sheet_name="Within-Anvil MW - roughness")
within_df_curvature = pd.read_excel(file_path, sheet_name="Within-Anvil MW - curvature")

# Anvil order by raw material group
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]
anvils = [a for a in anvil_order if a in within_df_roughness["anvil_id"].values]

within_df_roughness = within_df_roughness.set_index("anvil_id").reindex(anvils).reset_index()
within_df_curvature = within_df_curvature.set_index("anvil_id").reindex(anvils).reset_index()

color_roughness = "#440154" 
color_curvature = "#21918c" 

sns.set_theme(style="white", context="talk")
fig, ax = plt.subplots(figsize=(16, 7))

y = np.arange(len(anvils))

ax.scatter(
    within_df_roughness["CLES"], y,
    facecolors=color_roughness, edgecolors=color_roughness,
    linewidths=1.5, s=60, zorder=3,
)
ax.scatter(
    within_df_curvature["CLES"], y,
    facecolors=color_curvature, edgecolors=color_curvature,
    linewidths=1.5, s=60, zorder=3,
)

for val in [0.25, 0.5, 0.75]:
    ax.axvline(val, color="lightgrey", linewidth=1, linestyle="-", zorder=1)

gap = 0.005

for i in range(len(anvils)):
    r_val   = within_df_roughness["CLES"].iloc[i]
    c_val   = within_df_curvature["CLES"].iloc[i]
    x_left  = min(r_val, c_val) + gap
    x_right = max(r_val, c_val) - gap
    if x_right > x_left:
        ax.plot([x_left, x_right], [y[i], y[i]],
                color="black", linewidth=0.5, zorder=2)

ax.spines["left"].set_linewidth(0.75)
ax.spines["bottom"].set_linewidth(0.75)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_yticks(y)
ax.set_yticklabels(anvils, fontsize=10)
ax.set_xlim(0, 1.0)
ax.xaxis.set_major_locator(plt.MaxNLocator(5))
ax.set_xlabel("Common Language Effect Size", fontsize=12, labelpad=10)
ax.set_ylabel("Anvil ID", fontsize=12, labelpad=10)
ax.tick_params(axis="both", pad=8, labelsize=10)
ax.set_title("Within-anvil effect size (damaged vs. undamaged)", fontsize=14, weight="normal")
ax.grid(axis="x", alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

legend_elements = [
    mpatches.Patch(facecolor=color_roughness, edgecolor=color_roughness, linewidth=1.5, label="Roughness"),
    mpatches.Patch(facecolor=color_curvature, edgecolor=color_curvature, linewidth=1.5, label="Curvature"),
]
ax.legend(
    handles=legend_elements,
    fontsize=11, frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    borderaxespad=0,
)

plt.tight_layout()
plt.show()

Generates a two-panel heatmap figure showing roughness and curvature data for damaged and undamaged coordinates per anvil.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


def reorient_patch_to_flat(patch_data):
    """
    rotate patch so local surface is horizontal (normal points up).
    uses pca to find the best-fit plane and rotates to align it with the xy plane.
    """
    xyz = patch_data[['X', 'Y', 'Z']].values
    pca = PCA(n_components=3)
    pca.fit(xyz)
    normal = pca.components_[2]
    z_axis = np.array([0, 0, 1])
    rotation_axis = np.cross(normal, z_axis)
    rotation_axis_norm = np.linalg.norm(rotation_axis)

    if rotation_axis_norm < 1e-6:
        rotation_matrix = np.eye(3)
    else:
        rotation_axis = rotation_axis / rotation_axis_norm
        cos_angle = np.dot(normal, z_axis)
        angle = np.arccos(np.clip(cos_angle, -1, 1))
        K = np.array([
            [0, -rotation_axis[2], rotation_axis[1]],
            [rotation_axis[2], 0, -rotation_axis[0]],
            [-rotation_axis[1], rotation_axis[0], 0]
        ])
        rotation_matrix = (np.eye(3) +
                           np.sin(angle) * K +
                           (1 - np.cos(angle)) * K @ K)

    xyz_rotated = xyz @ rotation_matrix.T
    xyz_centered = xyz_rotated - xyz_rotated.mean(axis=0)
    return xyz_centered[:, 0], xyz_centered[:, 1]

# Path to the folder containing one folder per island locality. Each locality folder contains one folder per anvil with an _output.txt file.
root_dir = Path("FILEPATH")

anvil_dirs = [d for d in root_dir.glob("*/*") if d.is_dir()]
print(f"Found {len(anvil_dirs)} anvil(s)")

cmap = 'viridis'
n_patches = 10

for anvil_dir in anvil_dirs:
    anvil_id = anvil_dir.name
    print(f"\nProcessing anvil: {anvil_id}")

    spatial_join_files = list(anvil_dir.glob("*_spatial_join.txt"))
    output_files       = list(anvil_dir.glob("*_output.txt"))

    if not spatial_join_files:
        print("no spatial_join file found – skipping")
        continue
    if not output_files:
        print("no output file found – skipping")
        continue

    spatial_df = pd.read_csv(spatial_join_files[0], sep=';', skiprows=1,
                             names=['X', 'Y', 'Z', 'R', 'G', 'B',
                                    'roughness_temp', 'curvature_temp',
                                    'Nx', 'Ny', 'Nz'])

    output_df = pd.read_csv(output_files[0], sep=';', skiprows=1,
                            names=['X', 'Y', 'Z', 'roughness', 'curvature', 'condition'])
    output_df['condition'] = output_df['condition'].str.strip().str.lower()

    print(f"  spatial join: {len(spatial_df)} pts  |  output: {len(output_df)} pts")

    damaged_output   = output_df[output_df['condition'] == 'damaged'].copy()
    undamaged_output = output_df[output_df['condition'] == 'undamaged'].copy()

    undamaged_merged = undamaged_output.merge(
        spatial_df[['X', 'Y', 'Z', 'R', 'G', 'B']],
        on=['X', 'Y', 'Z'], how='inner')

    spatial_xyz = spatial_df[['X', 'Y', 'Z']].values
    damaged_xyz = damaged_output[['X', 'Y', 'Z']].values
    tree        = cKDTree(spatial_xyz)
    distances, indices = tree.query(damaged_xyz, k=1)
    damaged_output['R'] = spatial_df.iloc[indices]['R'].values
    damaged_output['G'] = spatial_df.iloc[indices]['G'].values
    damaged_output['B'] = spatial_df.iloc[indices]['B'].values

    merged_df    = pd.concat([damaged_output, undamaged_merged], ignore_index=True)
    damaged_df   = merged_df[merged_df['condition'] == 'damaged']
    undamaged_df = merged_df[merged_df['condition'] == 'undamaged'].copy()

    kmeans = KMeans(n_clusters=n_patches, random_state=42)
    undamaged_df['patch_id'] = kmeans.fit_predict(undamaged_df[['X', 'Y', 'Z']])
    patch_centers = kmeans.cluster_centers_

    sample_idx = np.random.choice(len(spatial_df),
                                  size=int(len(spatial_df) * 0.3),
                                  replace=False)
    spatial_sample = spatial_df.iloc[sample_idx]
    rgb_colors = np.column_stack([
        spatial_sample['R'].values / 255.0,
        spatial_sample['G'].values / 255.0,
        spatial_sample['B'].values / 255.0,
    ])

    # Percentile colour limits for readability.
    rough_vmin = np.percentile(merged_df['roughness'], 5)
    rough_vmax = np.percentile(merged_df['roughness'], 95)
    curv_vmin  = np.percentile(merged_df['curvature'], 5)
    curv_vmax  = np.percentile(merged_df['curvature'], 95)

    fig = plt.figure(figsize=(18, 22))
    outer_gs = GridSpec(2, 1, figure=fig, hspace=0.10)

    metrics = [
        ('roughness', rough_vmin, rough_vmax, 'Roughness',      'Roughness [mm]'),
        ('curvature', curv_vmin,  curv_vmax,  'Mean curvature',  'Mean curvature'),
    ]

    for row_idx, (metric, vmin, vmax, title_sfx, cbar_lbl) in enumerate(metrics):

        inner_gs = GridSpecFromSubplotSpec(
            5, 5,
            subplot_spec=outer_gs[row_idx],
            wspace=0.02,
            hspace=0.30,
            width_ratios=[1, 1, 0.45, 0.45, 0.06],
        )

        ax_main = fig.add_subplot(inner_gs[:, :2])
        ax_main.set_title(f"{anvil_id} – {title_sfx}", fontsize=14)

        ax_main.scatter(spatial_sample['X'], spatial_sample['Y'],
                        c=rgb_colors, s=0.2, alpha=1.0)

        if len(damaged_df) > 0:
            ax_main.scatter(damaged_df['X'], damaged_df['Y'],
                            c=damaged_df[metric], cmap=cmap,
                            s=1, alpha=0.7, edgecolors='none',
                            vmin=vmin, vmax=vmax)

        # Apply a 3x3 gridl: 2 vertica + 2 horizontal evenly-spaced lines.
        for x in np.linspace(spatial_sample['X'].min(), spatial_sample['X'].max(), 4)[1:-1]:
            ax_main.axvline(x, color='gold', linewidth=1.5, linestyle='-', alpha=0.8)
        for y in np.linspace(spatial_sample['Y'].min(), spatial_sample['Y'].max(), 4)[1:-1]:
            ax_main.axhline(y, color='gold', linewidth=1.5, linestyle='-', alpha=0.8)

        for i, center in enumerate(patch_centers):
            ax_main.text(center[0], center[1], str(i + 1),
                         fontsize=9, ha='center', va='center',
                         color='gold', fontweight='bold')

        ax_main.set_xlabel('X (mm)')
        ax_main.set_ylabel('Y (mm)')
        ax_main.set_aspect('equal')
        ax_main.set_xlim(spatial_sample['X'].min() - 10,
                         spatial_sample['X'].max() + 10)
        ax_main.set_ylim(spatial_sample['Y'].min() - 10,
                         spatial_sample['Y'].max() + 10)

        for patch_idx in range(n_patches):
            patch_data = undamaged_df[undamaged_df['patch_id'] == patch_idx].copy()
            r = patch_idx // 2      
            c = 2 + (patch_idx % 2) 

            ax_p = fig.add_subplot(inner_gs[r, c])

            if len(patch_data) > 0:
                x_flat, y_flat = reorient_patch_to_flat(patch_data)
                ax_p.scatter(x_flat, y_flat,
                             c=patch_data[metric].values, cmap=cmap,
                             s=5, edgecolors='none',
                             vmin=vmin, vmax=vmax)
                ax_p.set_aspect('equal')

            ax_p.margins(y=0.15)
            ax_p.set_title(str(patch_idx + 1), fontsize=11, pad=4, color='black')
            ax_p.axis('off')

        cax = fig.add_subplot(inner_gs[:, 4])
        sm  = plt.cm.ScalarMappable(
            cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax)
        cb.set_label(cbar_lbl, rotation=270, labelpad=16)

    output_path = anvil_dir / f"{anvil_id}_panel_layout.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path.name}")
    plt.close()

print("\nprocessing complete!")